In [35]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math
import torch.optim as optim

In [66]:
class SimpleRNN(nn.Module):
  def __init__(self, input_size=1, hidden_size=32, output_size=1):
    super().__init__()
    self.hidden_size = hidden_size

    self.rnn = nn.RNN(
        input_size=input_size,
        hidden_size=hidden_size,
        batch_first=True
    )

    self.fc = nn.Linear(
        in_features=hidden_size,
        out_features=output_size
    )

  def forward(self, x):
    out,  h_n = self.rnn(x)
    out = out[:, -1, :]
    out = self.fc(out)
    return out


In [67]:
class SineDataSet(Dataset):
  def __init__(self, seq_len=20, num_samples=1000, noise_std=0.0):
    self.data = []
    self.targets = []
    self.seq_len = seq_len
    self.noise_std = noise_std

    for _ in range(num_samples):
      start = torch.rand(1).item() * 2 * math.pi

      x = torch.tensor([
          math.sin(start + j * 0.1) + noise_std * torch.rand(1).item()
          for j in range(seq_len)
      ])

      y = torch.tensor([math.sin(start + seq_len * 0.1)])
      self.data.append(x.unsqueeze(-1))
      self.targets.append(y)

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx], self.targets[idx]


In [68]:
seq_len = 20
train_dataset = SineDataSet(seq_len=seq_len, num_samples=1000, noise_std=20)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataset = SineDataSet(seq_len=seq_len, num_samples=100, noise_std=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [69]:
model = SimpleRNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epoch = 10

for epoch in range(num_epoch):
  model.train()
  running_loss = 0.0
  for batch_idx, (inputs, labels) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(inputs.float())
    loss = criterion(output,  labels.float())
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f"Epoch {epoch+1}/{num_epoch}, Loss: {running_loss/len(train_loader)}")

print("finish training")

Epoch 1/10, Loss: 0.5000353483926683
Epoch 2/10, Loss: 0.48585250737175106
Epoch 3/10, Loss: 0.4820807673155315
Epoch 4/10, Loss: 0.478282363168777
Epoch 5/10, Loss: 0.47415604903584435
Epoch 6/10, Loss: 0.47636517007199547
Epoch 7/10, Loss: 0.4623054336933863
Epoch 8/10, Loss: 0.462174339190362
Epoch 9/10, Loss: 0.4539438405680278
Epoch 10/10, Loss: 0.45168725223768325
finish training


In [72]:
model.eval()
test_loss = 0.0

with torch.no_grad():
  for batch_idx, (input, label) in enumerate(test_loader):
    output = model(input.float())
    loss = criterion(output, label.float())
    test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader)}")

Test Loss: 0.6227166865553174
